# Sales Forecasting
## Time Series Analysis & Prediction

---

**Author:** Data Science Team  
**Date:** June 2026  
**Objective:** Forecast future sales using time series analysis and machine learning

## 1. Import Libraries

Import all necessary libraries for time series analysis and forecasting.

In [ ]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Time Series
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Warning handling
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('All libraries imported successfully!')

## 2. Data Loading

Load the sales dataset.

In [ ]:
# Generate synthetic sales data
np.random.seed(42)
n_days = 730  # 2 years of data

# Create date range
dates = pd.date_range(start='2024-01-01', periods=n_days, freq='D')

# Generate sales with patterns
trend = np.linspace(1000, 1500, n_days)  # Upward trend
seasonal = 200 * np.sin(2 * np.pi * np.arange(n_days) / 365.25)  # Yearly seasonality
weekly = 100 * np.sin(2 * np.pi * np.arange(n_days) / 7)  # Weekly pattern
noise = np.random.normal(0, 50, n_days)  # Random noise

# Combine components
sales = trend + seasonal + weekly + noise
sales = np.maximum(sales, 100)  # Ensure positive values

# Create additional features
product_category = np.random.choice(['Electronics', 'Clothing', 'Home', 'Food'], n_days)
promotion = np.random.choice([0, 1], n_days, p=[0.7, 0.3])
is_holiday = np.random.choice([0, 1], n_days, p=[0.95, 0.05])

# Create DataFrame
df = pd.DataFrame({
    'date': dates,
    'sales': sales.round(2),
    'product_category': product_category,
    'promotion': promotion,
    'is_holiday': is_holiday
})

# Set date as index
df.set_index('date', inplace=True)

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
print(f'\nFirst 5 rows:')
df.head()

## 3. Exploratory Data Analysis (EDA)

Analyze sales patterns, trends, and seasonality.

In [ ]:
# Basic information
print('='*60)
print('DATASET INFORMATION')
print('='*60)
print(f'\nShape: {df.shape}')
print(f'\nBasic Statistics:\n{df["sales"].describe()}')
print(f'\nTotal Sales: ${df["sales"].sum():,.2f}')
print(f'Average Daily Sales: ${df["sales"].mean():,.2f}')

In [ ]:
# Sales over time
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Daily sales
axes[0, 0].plot(df.index, df['sales'], color='#3498db', alpha=0.7)
axes[0, 0].set_title('Daily Sales Over Time', fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Sales ($)')

# Monthly average sales
monthly_sales = df['sales'].resample('M').mean()
axes[0, 1].plot(monthly_sales.index, monthly_sales.values, marker='o', color='#e74c3c')
axes[0, 1].set_title('Monthly Average Sales', fontweight='bold')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Average Sales ($)')

# Sales distribution
axes[1, 0].hist(df['sales'], bins=50, color='#2ecc71', edgecolor='black')
axes[1, 0].set_title('Sales Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Sales ($)')
axes[1, 0].set_ylabel('Frequency')

# Sales by day of week
df['day_of_week'] = df.index.dayofweek
daily_avg = df.groupby('day_of_week')['sales'].mean()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1, 1].bar(days, daily_avg.values, color='#9b59b6')
axes[1, 1].set_title('Average Sales by Day of Week', fontweight='bold')
axes[1, 1].set_xlabel('Day of Week')
axes[1, 1].set_ylabel('Average Sales ($)')

plt.tight_layout()
plt.show()

## 4. Time Series Decomposition

Decompose the time series into trend, seasonal, and residual components.

In [ ]:
# Seasonal decomposition
decomposition = seasonal_decompose(df['sales'], model='additive', period=30)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))

decomposition.observed.plot(ax=axes[0], title='Observed')
decomposition.trend.plot(ax=axes[1], title='Trend')
decomposition.seasonal.plot(ax=axes[2], title='Seasonal')
decomposition.resid.plot(ax=axes[3], title='Residual')

plt.tight_layout()
plt.show()

In [ ]:
# Augmented Dickey-Fuller test for stationarity
result = adfuller(df['sales'].dropna())

print('Augmented Dickey-Fuller Test')
print('='*50)
print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value: {result[1]:.4f}')
print(f'Critical Values:')
for key, value in result[4].items():
    print(f'  {key}: {value:.4f}')

if result[1] < 0.05:
    print('\nConclusion: The series is stationary (reject null hypothesis)')
else:
    print('\nConclusion: The series is non-stationary (fail to reject null hypothesis)')

## 5. Feature Engineering

Create features for machine learning models.

In [ ]:
# Create time-based features
df_features = df.copy()

# Time features
df_features['year'] = df_features.index.year
df_features['month'] = df_features.index.month
df_features['day'] = df_features.index.day
df_features['day_of_week'] = df_features.index.dayofweek
df_features['day_of_year'] = df_features.index.dayofyear
df_features['week_of_year'] = df_features.index.isocalendar().week.astype(int)

# Lag features
df_features['sales_lag_1'] = df_features['sales'].shift(1)
df_features['sales_lag_7'] = df_features['sales'].shift(7)
df_features['sales_lag_30'] = df_features['sales'].shift(30)

# Rolling statistics
df_features['sales_rolling_mean_7'] = df_features['sales'].rolling(window=7).mean()
df_features['sales_rolling_mean_30'] = df_features['sales'].rolling(window=30).mean()
df_features['sales_rolling_std_7'] = df_features['sales'].rolling(window=7).std()

# Drop rows with NaN values
df_features = df_features.dropna()

print(f'Features created. Shape: {df_features.shape}')
print(f'\nNew columns: {list(df_features.columns)}')

## 6. Model Building

Train multiple models for sales forecasting.

In [ ]:
# Prepare features
feature_cols = ['year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year',
                'sales_lag_1', 'sales_lag_7', 'sales_lag_30',
                'sales_rolling_mean_7', 'sales_rolling_mean_30', 'sales_rolling_std_7',
                'promotion', 'is_holiday']

X = df_features[feature_cols]
y = df_features['sales']

# Train-test split (time-based)
split_idx = int(len(df_features) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set: {X_test.shape[0]} samples')

# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate
results = {}

print('\n' + '='*70)
print('MODEL TRAINING AND EVALUATION')
print('='*70)

for name, model in models.items():
    print(f'\nTraining {name}...')
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2,
        'y_pred': y_pred
    }
    
    print(f'\n{name} Results:')
    print(f'  MAE:  ${mae:,.2f}')
    print(f'  RMSE: ${rmse:,.2f}')
    print(f'  MAPE: {mape:.2f}%')
    print(f'  R²:   {r2:.4f}')

## 7. Model Comparison

Compare model performance and visualize predictions.

In [ ]:
# Comparison dataframe
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'MAE': [results[m]['MAE'] for m in results],
    'RMSE': [results[m]['RMSE'] for m in results],
    'MAPE (%)': [results[m]['MAPE'] for m in results],
    'R² Score': [results[m]['R2'] for m in results]
})

print('\n' + '='*60)
print('MODEL COMPARISON')
print('='*60)
print(comparison_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Actual vs Predicted
best_model_name = max(results, key=lambda x: results[x]['R2'])
axes[0, 0].plot(y_test.index, y_test.values, label='Actual', color='#2ecc71')
axes[0, 0].plot(y_test.index, results[best_model_name]['y_pred'], 
                label='Predicted', color='#e74c3c', linestyle='--')
axes[0, 0].set_title(f'Actual vs Predicted - {best_model_name}', fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Sales ($)')
axes[0, 0].legend()

# Prediction error distribution
errors = y_test - results[best_model_name]['y_pred']
axes[0, 1].hist(errors, bins=50, color='#3498db', edgecolor='black')
axes[0, 1].set_title('Prediction Error Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Error ($)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(x=0, color='r', linestyle='--')

# Model metrics comparison
comparison_df.set_index('Model')[['MAE', 'RMSE']].plot(kind='bar', ax=axes[1, 0], colormap='viridis')
axes[1, 0].set_title('Model Error Metrics', fontweight='bold')
axes[1, 0].set_ylabel('Error ($)')
axes[1, 0].tick_params(axis='x', rotation=0)

# R² comparison
comparison_df.set_index('Model')[['R² Score']].plot(kind='bar', ax=axes[1, 1], color='#2ecc71')
axes[1, 1].set_title('Model R² Scores', fontweight='bold')
axes[1, 1].set_ylabel('R² Score')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 8. Feature Importance

Analyze which features are most important for predictions.

In [ ]:
# Feature importance from Random Forest
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('\nFeature Importance (Random Forest):')
print(feature_importance.to_string(index=False))

# Plot feature importance
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance for Sales Forecasting', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 9. Future Sales Prediction

Use the model to predict future sales.

In [ ]:
# Generate future dates for prediction
future_dates = pd.date_range(start=df.index[-1] + pd.Timedelta(days=1), periods=30, freq='D')

# Create future features (using last known values for lag features)
future_df = pd.DataFrame(index=future_dates)
future_df['year'] = future_df.index.year
future_df['month'] = future_df.index.month
future_df['day'] = future_df.index.day
future_df['day_of_week'] = future_df.index.dayofweek
future_df['day_of_year'] = future_df.index.dayofyear
future_df['week_of_year'] = future_df.index.isocalendar().week.astype(int)

# Use last known values for lag features
future_df['sales_lag_1'] = df['sales'].iloc[-1]
future_df['sales_lag_7'] = df['sales'].iloc[-7]
future_df['sales_lag_30'] = df['sales'].iloc[-30]
future_df['sales_rolling_mean_7'] = df['sales'].iloc[-7:].mean()
future_df['sales_rolling_mean_30'] = df['sales'].iloc[-30:].mean()
future_df['sales_rolling_std_7'] = df['sales'].iloc[-7:].std()
future_df['promotion'] = 0  # Default
future_df['is_holiday'] = 0  # Default

# Predict future sales
future_predictions = rf_model.predict(future_df[feature_cols])

# Create forecast dataframe
forecast_df = pd.DataFrame({
    'date': future_dates,
    'predicted_sales': future_predictions.round(2)
})

print('\n' + '='*60)
print('30-DAY SALES FORECAST')
print('='*60)
print(forecast_df.to_string(index=False))
print(f'\nTotal Predicted Sales (30 days): ${forecast_df["predicted_sales"].sum():,.2f}')
print(f'Average Daily Predicted Sales: ${forecast_df["predicted_sales"].mean():,.2f}')

In [ ]:
# Visualize forecast
plt.figure(figsize=(14, 6))

# Historical sales (last 90 days)
plt.plot(df.index[-90:], df['sales'].iloc[-90:], label='Historical Sales', color='#3498db')

# Forecasted sales
plt.plot(forecast_df['date'], forecast_df['predicted_sales'], 
         label='Forecasted Sales', color='#e74c3c', linestyle='--', marker='o')

plt.axvline(x=df.index[-1], color='gray', linestyle=':', label='Forecast Start')
plt.fill_between(forecast_df['date'], 
                 forecast_df['predicted_sales'] * 0.9, 
                 forecast_df['predicted_sales'] * 1.1, 
                 alpha=0.2, color='red', label='Confidence Interval')

plt.title('30-Day Sales Forecast', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Conclusions & Recommendations

### Key Findings:
1. **Trend**: Sales show a clear upward trend over time
2. **Seasonality**: Strong weekly and monthly seasonal patterns exist
3. **Lag Features**: Previous sales values are strong predictors
4. **Rolling Statistics**: Moving averages capture local trends effectively

### Recommendations:
1. **Inventory Planning**: Use forecasts for inventory management
2. **Staffing**: Adjust staffing based on predicted busy periods
3. **Promotions**: Time promotions during predicted slow periods
4. **Budgeting**: Use sales forecasts for financial planning

In [ ]:
# Save the best model
import joblib
import os

# Create models directory
os.makedirs('models', exist_ok=True)

# Save model
joblib.dump(rf_model, 'models/sales_forecast_model.pkl')

print(f'Best model (Random Forest) saved successfully!')
print(f'\nFinal Model Performance:')
print(f'  - MAE: ${results["Random Forest"]["MAE"]:,.2f}')
print(f'  - RMSE: ${results["Random Forest"]["RMSE"]:,.2f}')
print(f'  - R² Score: {results["Random Forest"]["R2"]:.4f}')